# DAE para deteccion de anomalias cerebrales

Notebook reproducible para Brain-AD / BraTS2021 (FLAIR) en Google Colab. Usa un unico modelo: un **Denoising Autoencoder (DAE)** sencillo, basado en [Kascenas et al., MIDL 2022](https://proceedings.mlr.press/v172/kascenas22a.html).

El DAE recibe imagenes normales con ruido gaussiano grueso, aprende a reconstruir la imagen limpia y usa el error de reconstruccion como puntuacion de anomalia. La red tiene forma de U-Net, conexiones skip y ningun cuello de botella restrictivo.

In [ ]:
from pathlib import Path
import json
import os
import platform
import shutil
import subprocess
import sys
import time

REPO_URL = "https://github.com/alerodriargui/TFMv3.git"
BRANCH = "brain"
PROJECT_DIR = Path("/content/TFMv3")
IMAGE_SIZE = 224
EPOCHS = 100
SEEDS = [42]
MOUNT_DRIVE = True
BACKUP_DIR = Path("/content/drive/MyDrive/TFMv3_colab_backup")

print("Python", sys.version.split()[0])
print("Platform", platform.platform())

## 1. Preparar el proyecto

In [ ]:
os.chdir("/content")
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)

from tfm_ae.dae import DAE
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
print("Modelo:", DAE.__name__)

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Activa Runtime > Change runtime type > GPU")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

## 2. Descargar y comprobar Brain-AD

In [ ]:
import gdown
import zipfile

ZIP_PATH = Path("/content/Brain_AD.zip")
ZIP_URL = "https://drive.google.com/uc?id=1xuapqiL1s18eMeKMd-Vi1Cqp2wp7Pghp"

def find_dataset_root():
    candidates = [Path("/content"), *Path("/content").iterdir()]
    for root in candidates:
        if not root.is_dir() or root.name == "drive":
            continue
        if (root / "train/good").is_dir() and (root / "valid/good").is_dir() and (root / "test").is_dir():
            return root
    return None

DATA_ROOT = find_dataset_root()
if DATA_ROOT is None:
    gdown.download(ZIP_URL, str(ZIP_PATH), quiet=False)
    if not zipfile.is_zipfile(ZIP_PATH):
        raise RuntimeError("La descarga de Brain_AD.zip no es valida")
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall("/content")
    DATA_ROOT = find_dataset_root()
if DATA_ROOT is None:
    raise FileNotFoundError("No se encontro train/valid/test")

os.environ["TFM_DATA_ROOT"] = str(DATA_ROOT)
print("Dataset:", DATA_ROOT)

In [ ]:
from tfm_ae.data import find_images, split_dir

folders = {
    "train/good": split_dir(DATA_ROOT, "train") / "good",
    "valid/good": split_dir(DATA_ROOT, "val") / "good",
    "valid/Ungood": split_dir(DATA_ROOT, "val") / "Ungood",
    "test/good": split_dir(DATA_ROOT, "test") / "good",
    "test/Ungood": split_dir(DATA_ROOT, "test") / "Ungood",
}
dataset_counts = {name: len(find_images(path)) for name, path in folders.items()}
if not all(dataset_counts.values()):
    raise RuntimeError(f"Dataset incompleto: {dataset_counts}")
print(json.dumps(dataset_counts, indent=2))

## 3. Prueba reducida

In [ ]:
from tfm_ae.experiment import ExperimentConfig, run

SMOKE_OUTPUT = Path("results/colab_smoke/dae_seed42")
shutil.rmtree(SMOKE_OUTPUT.parent, ignore_errors=True)
smoke = run(ExperimentConfig(
    data_root=DATA_ROOT,
    output_dir=SMOKE_OUTPUT,
    epochs=1,
    batch_size=4,
    image_size=IMAGE_SIZE,
    max_train_images=32,
    max_eval_images_per_class=16,
))
print(json.dumps(smoke["test"], indent=2))
assert smoke["device"] == "cuda"

## 4. Entrenamiento DAE

In [ ]:
FULL_OUTPUT = Path("results/colab_dae_full")
FULL_OUTPUT.mkdir(parents=True, exist_ok=True)
reports = []
started = time.perf_counter()

for seed in SEEDS:
    print(f"Entrenando DAE seed={seed}")
    report = run(ExperimentConfig(
        data_root=DATA_ROOT,
        output_dir=FULL_OUTPUT / f"dae_seed{seed}",
        epochs=EPOCHS,
        batch_size=16,
        image_size=IMAGE_SIZE,
        seed=seed,
    ))
    reports.append(report)
    if MOUNT_DRIVE:
        BACKUP_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copytree(FULL_OUTPUT, BACKUP_DIR / "results_colab_dae_full", dirs_exist_ok=True)

elapsed = time.perf_counter() - started
print(f"Duracion total: {elapsed:.1f} s")

In [ ]:
import pandas as pd

summary = pd.DataFrame([{
    "seed": report["config"]["seed"],
    "epoch": report["selected_epoch"],
    "auroc": report["test"]["auroc"],
    "balanced_accuracy": report["test"]["balanced_accuracy"],
} for report in reports])
display(summary)

## 5. Descargar artefactos

In [ ]:
from google.colab import files

archive = shutil.make_archive("/content/TFMv3_colab_dae", "zip", root_dir=FULL_OUTPUT)
print("Artefactos:", archive)
files.download(archive)